# Data Preprocessing

## Import Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from imblearn.under_sampling import RandomUnderSampler

# Load repo into /content from filessystem

In [2]:
# clone repo and move into it

!git clone https://github.com/learnbh/ml-project.git
%cd ml-project

# check files
!ls

Cloning into 'ml-project'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 91 (delta 24), reused 81 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 13.85 MiB | 17.57 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/ml-project
bea_eda.ipynb		       eike_eda.ipynb  README.md
bea_eda_test.ipynb	       example_files   README-NeueFische.md
data			       images	       requirements.txt
EDA-and-modeling.ipynb	       LICENSE	       results
EDA_BaselineModel.ipynb        Makefile
eike_data_preprocessing.ipynb  models


In [3]:
from google.colab import drive
import os
import shutil

# 1. Drive mounten
drive.mount('/content/drive')

# 2. Pfad zu deinem Ordner (JETZT korrekt!)
drive_path = '/content/drive/MyDrive/NeueFische/ml_project/data'

# 3. Zielordner im Colab-Environment
target_path = '/content/ml-project/data'
os.makedirs(target_path, exist_ok=True)

# 4. Dateien kopieren
for f in os.listdir(drive_path):
    src = os.path.join(drive_path, f)
    dst = os.path.join(target_path, f)

    if os.path.isfile(src):   # nur Dateien, keine Unterordner
        shutil.copy(src, dst)

print("Files copied!")

# 5. Check
!ls /content/data

Mounted at /content/drive
Files copied!
ls: cannot access '/content/data': No such file or directory


## Read the Data

In [5]:
client_train = pd.read_csv('data/client_train.csv', low_memory=False)
invoice_train = pd.read_csv('data/invoice_train.csv', low_memory=False)

client_test = pd.read_csv(f'data/client_test.csv', low_memory=False)
invoice_test = pd.read_csv(f'data/invoice_test.csv', low_memory=False)
#sample_submission = pd.read_csv(f'results/SampleSubmission.csv', low_memory=False)

In [34]:
client_train.head()

,disrict,client_id,client_catg,region,creation_date,target
75532,69,train_Client_46032,11,105,25/12/2015,0.0
123094,69,train_Client_8884,11,107,13/11/2010,1.0
114229,60,train_Client_80860,11,101,28/10/2008,0.0
48514,60,train_Client_21716,11,101,29/10/1984,0.0
130130,62,train_Client_95171,11,301,24/03/1989,0.0


## Data Cleaning

In [6]:
train = invoice_train.copy()
# counter_statue == 269375 -> can be droped, since there is just one observation and also the train_Client_53725 belong to only that observation
train = train.query("counter_statue != 269375")
# counter_statue == 420 and reading_remarque == 5 and counter_code == 1 -> they are just one time in the same observation, so we can drop them as well
train = train.query("counter_statue != 420")
# replace string of int to int
train['counter_statue'] = train['counter_statue'].replace('0', 0)
train['counter_statue'] = train['counter_statue'].replace('1', 1)
train['counter_statue'] = train['counter_statue'].replace('4', 4)
train['counter_statue'] = train['counter_statue'].replace('5', 5)
# counter_statue == 769 and reading_remarque == 207 ???
train = train.query("counter_statue != 769")
# counter_statue == 618 and reading_remarque == 413 ???
train = train.query("counter_statue != 618")
# counter_statue == 46 and reading_remarque == 203 ???
train = train.query("counter_statue != 46")
#months_number > 88
train = train.query("months_number <= 88") # auch aus testdaten
invoice_train = train.copy()

invoice_test = invoice_test.query("months_number <= 88")

## Train Test Split

In [7]:
# split client train, stratify by target variable, and set random state for reproducibility
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(client_train.drop(columns=['target']), client_train['target'], test_size=0.2, stratify=client_train['target'], random_state=42)

In [8]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)

(108394, 5) (108394,)
(27099, 5) (27099,)


In [9]:
client_train = pd.merge(X_train, y_train, left_index=True, right_index=True)
client_val = pd.merge(X_val, y_val, left_index=True, right_index=True)

### undersampling

In [10]:
# undersample the majority class in the training data
rus = RandomUnderSampler(random_state=42)
X_train_res, y_train_res = rus.fit_resample(X_train, y_train)

In [11]:
clients_in_resampled_train = X_train_res['client_id']
clients_in_val = X_val['client_id']

invoice_train_res = invoice_train[invoice_train['client_id'].isin(clients_in_resampled_train)]
invoice_val = invoice_train[invoice_train['client_id'].isin(clients_in_val)]

client_train_res = pd.merge(X_train_res, y_train_res, left_index=True, right_index=True)

In [12]:
print(X_train_res.shape, y_train_res.shape)
print(X_val.shape, y_val.shape)

(12106, 5) (12106,)
(27099, 5) (27099,)


## Feature Engineering Functions

In [13]:
#convert the column invoice_date to date time format on both the invoice train and invoice test
def invoice_convert_date(df):
    df_updated = df.copy()
    df_updated['invoice_date'] = pd.to_datetime(df_updated['invoice_date'])
    return df_updated

In [14]:
# convert tarif_type, counter_statue, counter_code, reading_remarque, counter_type to categorical data type
def invoice_to_category(df):
    df_updated = df.copy()
    df_updated['tarif_type'] = df_updated['tarif_type'].astype('category')
    df_updated['counter_statue'] = df_updated['counter_statue'].astype('category')
    df_updated['counter_code'] = df_updated['counter_code'].astype('category')
    df_updated['reading_remarque'] = df_updated['reading_remarque'].astype('category')
    df_updated['counter_type'] = df_updated['counter_type'].astype('category')
    return df_updated

In [29]:
# convert disrict, client_catg, region to categorical data type
def client_to_category(df):
    df_updated = df.copy()
    df_updated['disrict'] = df_updated['disrict'].astype('category')
    df_updated['client_catg'] = df_updated['client_catg'].astype('category')
    df_updated['region'] = df_updated['region'].astype('category')
    return df_updated

In [15]:
# create variable that sums the consumption for all 4 levels
def invoice_create_consumption(df):
    df_updated = df.copy()
    df_updated['total_consumption'] = df_updated['new_index'] - df_updated['old_index']
    df_updated['consumption_per_month'] = df_updated['total_consumption'] / df_updated['months_number']
    return df_updated

In [16]:
# convert creation_date to date time format on both the client train and client test
def client_convert_date(df):
    df_updated = df.copy()
    df_updated['creation_date'] = pd.to_datetime(df_updated['creation_date'], dayfirst=True)
    # create new variable with 7 bins for the creation date using cut
    df_updated['creation_date_bin'] = pd.cut(df_updated['creation_date'], bins=7, labels=False)
    # convert the creation_date_bin to categorical data type
    df_updated['creation_date_bin'] = df_updated['creation_date_bin'].astype('category')
    return df_updated

In [17]:
# DEFINE LEVEL USAGE
# -------------------------
def define_activity_level4_usage(df):
    df_updated = df.copy()
    df_updated['used_lvl4'] = (df_updated['consommation_level_4'] > 0).astype(int)
    # activity
    # any consumption at all (levels 1–4)
    df_updated['active'] = (
        (df_updated['consommation_level_1'] > 0) |
        (df_updated['consommation_level_2'] > 0) |
        (df_updated['consommation_level_3'] > 0) |
        (df_updated['consommation_level_4'] > 0)
    ).astype(int)
    return df_updated

In [24]:
def duplicate_invoices_same_day(df):
    df_updated = df.copy()
    dup_counts = df_updated.groupby(
        ['client_id', 'counter_type', 'invoice_date']
    ).size().reset_index(name='n_invoices')
    dup_counts['multi_flag'] = (dup_counts['n_invoices'] > 1).astype(int)
    client_multi_flag = dup_counts.groupby('client_id')['multi_flag'].max()
    # map to df
    df_updated['has_multi_invoice_same_day'] = df_updated['client_id'].map(client_multi_flag)
    return df_updated

In [23]:
# date gaps
def date_gaps(df):
    df_updated = df.copy()
    df_updated = df_updated.sort_values(['client_id', 'invoice_date'])
    df_updated['gap_days'] = df_updated.groupby('client_id')['invoice_date'].diff().dt.days
    return df_updated

In [18]:
# -------------------------
# FRACTION PER CLIENT
# -------------------------
def lvl4_fraction(df):
    df_updated = df.copy()
    client_lvl4_fraction = df_updated.groupby('client_id')['used_lvl4'].mean().reset_index()
    client_lvl4_fraction.rename(columns={
        'used_lvl4': 'frac_time_lvl4'
        }, inplace=True)
    df_updated['frac_time_lvl4'] = df_updated['client_id'].map(
        client_lvl4_fraction.set_index('client_id')['frac_time_lvl4']
    )
    return df_updated

In [19]:
def invoice_scaled_consumption_per_month(df):
    df_updated = df.copy()
    df_updated['consumption_per_month_scaled'] = df_updated.groupby(['client_id', 'counter_type'])['consumption_per_month'].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0.5)
    return df_updated

In [20]:
def correct_newIndex(df):
    updated_df = df.copy()
    updated_df['index_diff'] = updated_df['new_index'] - updated_df['old_index']
    updated_df.loc[updated_df['index_diff'] < 0, 'new_index'] += 100000
    return updated_df

In [21]:
# df = invoice_train.copy()

# # level 4 used or not
# df['lvl4'] = (df['consommation_level_4'] > 0).astype(int)

# # date
# df['invoice_date'] = pd.to_datetime(df['invoice_date'])

# # -------------------------
# # FIX: collapse per client + counter + date
# # -------------------------
# df_day = df.groupby(
#     ['client_id', 'counter_number', 'invoice_date']
# )['lvl4'].max().reset_index()

# -------------------------
# sort by time
# -------------------------
# df_day = df_day.sort_values(['client_id','counter_number','invoice_date'])

# previous value (per counter!)
# df_day['prev_lvl4'] = df_day.groupby(
#     ['client_id','counter_number']
# )['lvl4'].shift(1)

# detect jump 0 → 1
# df_day['jump_1_to_4'] = (
#     (df_day['prev_lvl4'] == 0) & (df_day['lvl4'] == 1)
# ).astype(int)

# -------------------------
# COUNT per client + counter
# -------------------------
# client_counter_jump_count = df_day.groupby(
#     ['client_id','counter_number']
# )['jump_1_to_4'].sum()

# client_counter_jump_count.shape

### Run on (undersampled) data

In [25]:
invoice_train_res = invoice_convert_date(invoice_train_res)
invoice_train_res = correct_newIndex(invoice_train_res)
invoice_train_res = invoice_to_category(invoice_train_res)
invoice_train_res = invoice_create_consumption(invoice_train_res)
invoice_train_res = define_activity_level4_usage(invoice_train_res)
invoice_train_res = lvl4_fraction(invoice_train_res)
invoice_train_res = duplicate_invoices_same_day(invoice_train_res)
invoice_train_res = date_gaps(invoice_train_res)
invoice_train_res = invoice_scaled_consumption_per_month(invoice_train_res)
# write the preprocessed invoice train_res to csv
invoice_train_res.to_csv('data/created_invoice_train_res.csv', index=False)

/tmp/ipykernel_2031/4269122176.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dup_counts = df_updated.groupby(
/tmp/ipykernel_2031/3021013401.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_updated['consumption_per_month_scaled'] = df_updated.groupby(['client_id', 'counter_type'])['consumption_per_month'].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0.5)


In [26]:
invoice_val = invoice_convert_date(invoice_val)
invoice_val = correct_newIndex(invoice_val)
invoice_val = invoice_to_category(invoice_val)
invoice_val = invoice_create_consumption(invoice_val)
invoice_val = define_activity_level4_usage(invoice_val)
invoice_val = lvl4_fraction(invoice_val)
invoice_val = duplicate_invoices_same_day(invoice_val)
invoice_val = date_gaps(invoice_val)
invoice_val = invoice_scaled_consumption_per_month(invoice_val)
# write the preprocessed invoice val to csv
invoice_val.to_csv('data/created_invoice_val.csv', index=False)

/tmp/ipykernel_2031/4269122176.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dup_counts = df_updated.groupby(
/tmp/ipykernel_2031/3021013401.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_updated['consumption_per_month_scaled'] = df_updated.groupby(['client_id', 'counter_type'])['consumption_per_month'].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0.5)


In [27]:
invoice_test = correct_newIndex(invoice_test)
invoice_test = invoice_convert_date(invoice_test)
invoice_test = invoice_to_category(invoice_test)
invoice_test = invoice_create_consumption(invoice_test)
invoice_test = define_activity_level4_usage(invoice_test)
invoice_test = lvl4_fraction(invoice_test)
invoice_test = duplicate_invoices_same_day(invoice_test)
invoice_test = date_gaps(invoice_test)
invoice_test = invoice_scaled_consumption_per_month(invoice_test)
# write the preprocessed invoice test to csv
invoice_test.to_csv('data/created_invoice_test.csv', index=False)

/tmp/ipykernel_2031/4269122176.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dup_counts = df_updated.groupby(
/tmp/ipykernel_2031/3021013401.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_updated['consumption_per_month_scaled'] = df_updated.groupby(['client_id', 'counter_type'])['consumption_per_month'].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0.5)


In [30]:
for df in [client_train_res, client_val, client_test]:
    df = client_convert_date(df)
    df = client_to_category(df)

client_train_res.to_csv('data/created_client_train_res.csv', index=False)
client_val.to_csv('data/created_client_val.csv', index=False)
client_test.to_csv('data/created_client_test.csv', index=False)

In [31]:
# read created train_res data
invoice_train_res = pd.read_csv('data/created_invoice_train_res.csv')
# read created data
invoice_val = pd.read_csv('data/created_invoice_val.csv')
# read created data
invoice_test = pd.read_csv('data/created_invoice_test.csv')

# read created client data
client_train_res = pd.read_csv('data/created_client_train_res.csv')
client_val = pd.read_csv('data/created_client_val.csv')
client_test = pd.read_csv('data/created_client_test.csv')

## Aggregate functions on invoice to client

In [35]:
def aggregate_by_client_id(invoice_data):
    aggs = {}
    aggs['gap_days'] = [np.nanmean]
    aggs['gap_days'] = [np.nanstd]
    aggs['consumption_per_month'] = [np.nanmean]
    aggs['consumption_per_month'] = [np.nanstd]
    aggs['frac_time_lvl4'] = [np.nanmax]
    aggs['consumption_per_month_scaled'] = [np.nanstd]
    aggs['active'] = ['sum']
    aggs['active'] = [np.nanmean]
    aggs['counter_statue'] = [np.nanstd]
    aggs['reading_remarque'] = [np.nanstd]
    aggs['tarif_type'] = [scipy.stats.mode]
    aggs['index_diff'] = [np.nanstd]


    agg_trans = invoice_data.groupby(['client_id']).agg(aggs)
    agg_trans.columns = ['_'.join(col).strip() for col in agg_trans.columns.values]
    agg_trans.reset_index(inplace=True)

    df = (invoice_data.groupby('client_id')
            .size()
            .reset_index(name='{}transactions_count'.format('1')))
    return pd.merge(df, agg_trans, on='client_id', how='left')

In [36]:
#group invoice data by client_id
agg_train = aggregate_by_client_id(invoice_train)
#group invoice data by client_id
agg_val = aggregate_by_client_id(invoice_val)
#group invoice data by client_id
agg_test = aggregate_by_client_id(invoice_test)

NameError: name 'scipy' is not defined

In [ ]:
print(agg_train.shape)
agg_train.head()

(135493, 6)


,client_id,1transactions_count,consommation_level_1_mean,consommation_level_2_mean,consommation_level_3_mean,consommation_level_4_mean
0,train_Client_0,35,352.400000,10.571429,0.000000,0.000000
1,train_Client_1,37,557.540541,0.000000,0.000000,0.000000
2,train_Client_10,18,798.611111,37.888889,0.000000,0.000000
3,train_Client_100,20,1.200000,0.000000,0.000000,0.000000
4,train_Client_1000,14,663.714286,104.857143,117.357143,36.714286


In [ ]:
#merge aggregate data with client dataset
train_res = pd.merge(client_train_res, agg_train, on='client_id', how='left')
display(train_res)

,disrict,client_id,client_catg,region,creation_date,target,1transactions_count,consommation_level_1_mean,consommation_level_2_mean,consommation_level_3_mean,consommation_level_4_mean
0,60,train_Client_0,11,101,31/12/1994,0.0,35,352.400000,10.571429,0.000000,0.000000
1,69,train_Client_1,11,107,29/05/2002,0.0,37,557.540541,0.000000,0.000000,0.000000
2,62,train_Client_10,11,301,13/03/1986,0.0,18,798.611111,37.888889,0.000000,0.000000
3,69,train_Client_100,11,105,11/07/1996,0.0,20,1.200000,0.000000,0.000000,0.000000
4,62,train_Client_1000,11,303,14/10/2014,0.0,14,663.714286,104.857143,117.357143,36.714286
...,...,...,...,...,...,...,...,...,...,...,...
135488,62,train_Client_99995,11,304,26/07/2004,0.0,71,1.957746,0.000000,0.000000,0.000000
135489,63,train_Client_99996,11,311,25/10/2012,0.0,41,185.853659,0.756098,0.000000,0.000000
135490,63,train_Client_99997,11,311,22/11/2011,0.0,36,273.083333,0.000000,0.000000,0.000000
135491,60,train_Client_99998,11,101,22/12/1993,0.0,2,300.000000,70.500000,0.000000,0.000000


In [ ]:
#aggregate test set
agg_test = aggregate_by_client_id(invoice_test)
test = pd.merge(client_test,agg_test, on='client_id', how='left')

In [ ]:
train.shape, test.shape

((135493, 11), (58069, 10))

In [ ]:
#drop redundant columns
sub_client_id = test['client_id']
drop_columns = ['client_id', 'creation_date']

for col in drop_columns:
    if col in train.columns:
        train.drop([col], axis=1, inplace=True)
    if col in test.columns:
        test.drop([col], axis=1, inplace=True)

## Tips
- Thorough EDA and incorporating domain knowledge
- Re-grouping categorical features
- More feature engineering(try utilizing some date-time features)
- Target balancing - oversampling, undersampling, SMOTE, scale_pos_weight
- Model ensembling
- Train-test split or cross-validation


# ******************* GOOD LUCK!!! ***************************